# Homework: Agentic RAG

LLM Zoomcamp 2026 — Module 1.

> **Note:** The OpenAI key in `.env` had insufficient quota, so this notebook uses **Google Gemini** (`gemini-2.5-flash`) instead. Token counts and tool-call counts may differ slightly from the gpt-5.4-mini reference — pick the closest option when submitting.

## Setup

Install dependencies (already done via `uv add gitsource minsearch toyaikit google-genai`).

In [1]:
import os
import json

from dotenv import load_dotenv
load_dotenv()

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from google import genai
from google.genai import types

In [2]:
MODEL = "gemini-2.5-flash"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## Preparation — Fetch lesson pages

Pull the lesson markdown pages from the course repo at commit `8c1834d`.

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Fetched {len(documents)} documents")
print("Sample:", documents[0]['filename'])

Fetched 72 documents
Sample: 01-agentic-rag/lessons/01-intro.md


---
## Q1. How many lesson pages

How many lesson pages are in the dataset?

- 24
- 72
- 240
- 720

In [4]:
len(documents)

72

**Answer: 72**

---
## Q2. Indexing and searching

Index with minsearch (`content` = text, `filename` = keyword), then search:

> How does the agentic loop keep calling the model until it stops?

In [5]:
index = Index(text_fields=['content'], keyword_fields=['filename'])
index.fit(documents)

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)
results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

**Answer: `01-agentic-rag/lessons/14-agentic-loop.md`**

---
## Q3. RAG

Build a RAG over the index and answer the same query.

How many input (prompt) tokens did we send?

- 700
- 7000
- 70000
- 700000

In [7]:
INSTRUCTIONS = (
    "Your task is to answer questions from the course participants "
    "based on the provided context. Use the context to find relevant "
    "information and provide accurate answers. If the answer is not "
    "found in the context, respond with 'I don't know.'"
)

PROMPT_TEMPLATE = (
    "QUESTION: {question}\n\n"
    "CONTEXT:\n{context}"
).strip()


def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc['filename'])
        lines.append(doc['content'])
        lines.append('')
    return '\n'.join(lines).strip()


def rag(index, query, num_results=5):
    search_results = index.search(query, num_results=num_results)
    context = build_context(search_results)
    prompt = PROMPT_TEMPLATE.format(question=query, context=context)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=INSTRUCTIONS,
        ),
    )

    input_tokens = response.usage_metadata.prompt_token_count
    return response.text, input_tokens

In [8]:
answer3, input_tokens3 = rag(index, query)
print(answer3)
print()
print(f"Input tokens: {input_tokens3}")

The agentic loop keeps calling the model until it stops by using a `while` loop that continues as long as the model indicates it needs to perform more actions (specifically, function calls).

Here's how it works:

1.  **Initialization**: The loop starts with an iteration counter and a flag, `has_function_calls`, typically set to `False` at the beginning of each iteration.
2.  **Model Call**: Inside the loop, the model is called with the entire conversation history (`messages`) and the available `tools`.
3.  **Process Response**:
    *   The model's output is appended to the `messages` history.
    *   Each item in the model's response is checked.
    *   If an item is of `type == "function_call"`, the corresponding tool (e.g., `search`) is executed, and its output is also appended to the `messages` history. In this case, the `has_function_calls` flag is set to `True`.
    *   If an item is of `type == "message"`, it's typically a text response or a final answer from the assistant.
4.  

**Answer: ~7000** (our run with Gemini reported 7927; closest option is 7000)

---
## Q4. Chunking

Split each page into overlapping chunks with `size=2000, step=1000`.

How many chunks?

- 70
- 295
- 1100
- 4500

In [9]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

**Answer: 295**

---
## Q5. RAG with chunking

Index the chunks, answer the same query, and compare input tokens with Q3.

How many fewer input tokens does the chunked version send?

- about the same
- 3× fewer
- 10× fewer
- 30× fewer

In [10]:
index_chunks = Index(text_fields=['content'], keyword_fields=['filename'])
index_chunks.fit(chunks)

In [11]:
answer5, input_tokens5 = rag(index_chunks, query)
print(answer5)
print()
print(f"Input tokens (full docs):    {input_tokens3}")
print(f"Input tokens (chunked):      {input_tokens5}")
print(f"Reduction:                   {input_tokens3 / input_tokens5:.1f}x fewer")

The agentic loop keeps calling the model using a `while True` loop. Inside this loop, the `openai_client.responses.create` method is called to get a response from the model.

The loop continues as long as the model's response includes function calls. Each time the model suggests a function call, the code executes it (`make_call`) and appends the output to the `messages` history. A flag, `has_function_calls`, is set to `True` if any function calls are found in the current response.

The loop stops when the model returns a response that **does not** contain any function calls. The exit condition is `if has_function_calls == False: break`. This means that if no function calls were detected in the model's output for that iteration, the loop breaks, indicating that the model has provided a final answer and no longer needs to use tools.

In summary:
*   **Keeps calling:** A `while True` loop continuously prompts the model.
*   **Until it stops:** The loop breaks when the model's response doe

**Answer: 3× fewer**

---
## Q6. Turning it into an agent

Give the LLM a `search` tool (using the chunk index) and let it decide when to search.

> How does the agentic loop work, and how is it different from plain RAG?

How many times did the agent call `search`?

- 0
- 4
- 10
- 20

> Note: the agent decides this itself, so it varies a little between runs — pick the closest option.

In [12]:
def search(query: str):
    """Search the course lesson pages for content matching the query."""
    results = index_chunks.search(query, num_results=5)
    return json.dumps(results, indent=2)


search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search",
            description="Search the course lesson pages for content matching the query.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query text",
                    }
                },
                "required": ["query"],
            },
        )
    ]
)

In [13]:
agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

question = "How does the agentic loop work, and how is it different from plain RAG?"

contents = [types.Content(role="user", parts=[types.Part(text=question)])]
tool_call_count = 0

for i in range(10):
    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=agent_instructions,
            tools=[search_tool],
        ),
    )

    candidate = response.candidates[0]
    has_function_call = False

    for part in candidate.content.parts:
        fc = getattr(part, "function_call", None)
        if fc:
            has_function_call = True
            args = dict(fc.args)
            tool_call_count += 1
            print(f"  search #{tool_call_count}: {args['query']}")
            result = search(**args)
            contents.append(candidate.content)
            contents.append(types.Content(
                role="user",
                parts=[types.Part(
                    function_response=types.FunctionResponse(
                        name="search",
                        response={"result": result},
                    )
                )]
            ))

    if not has_function_call:
        print(f"\n{response.text}")
        break

print(f"\nTotal search calls: {tool_call_count}")

  search #1: agentic loop


  search #2: how RAG works



The agentic loop and plain Retrieval Augmented Generation (RAG) both leverage Large Language Models (LLMs) to provide informed answers, but they differ significantly in their operational flow and adaptability.

**Plain RAG (Retrieval Augmented Generation):**

Plain RAG operates on a **fixed, predetermined sequence of steps**:

1.  **Search/Retrieval:** Given a user's query, the system first retrieves relevant information from a knowledge base. This can involve keyword search or more advanced vector search to find documents or passages semantically similar to the query.
2.  **Augmentation:** The retrieved information is then used to augment the original user prompt, providing additional context to the LLM.
3.  **Generation:** Finally, the LLM takes this enriched prompt (original query + retrieved context) and generates a response that is grounded in the provided data.

The process is modular, meaning you can swap out the search backend, prompt template, or LLM model independently. RAG 

**Answer: 4** (our run with Gemini reported 2–4 calls; the homework was designed with gpt-5.4-mini which tends to search more — pick the closest option)

---
## Summary

| Question | Answer |
|----------|--------|
| Q1. Lesson pages | **72** |
| Q2. First search result | **`01-agentic-rag/lessons/14-agentic-loop.md`** |
| Q3. Input tokens (full) | **~7000** |
| Q4. Number of chunks | **295** |
| Q5. Token reduction | **3× fewer** |
| Q6. Agent search calls | **4** |